# hparam-precedence-merge — worked example 2: Per-group hparam: group > kwarg > default precedence

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `hparam-precedence-merge`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Optimizer parameter groups can specify per-layer learning rates, weight decays, etc. The resolution order is: `group` dict wins over the optimizer-level `kwargs`, which win over the hardcoded `defaults`. This is the `{**defaults, **kwargs, **group}` pattern — each layer is a dict that shadows the previous one, and the final resolved value for any key is the one from the highest-priority dict that contains it.

## Worked solution

**Step 1 — define defaults, optimizer kwargs, and a group override.** Typical defaults include `lr=1e-2`, `weight_decay=0`. The optimizer was constructed with `lr=1e-3`. The specific parameter group says `lr=1e-4`.

**Step 2 — resolve with `{**defaults, **kwargs, **group}`.** The group's `lr=1e-4` beats the kwarg's `lr=1e-3`, which beats the default's `lr=1e-2`.

**Step 3 — check that missing keys fall through correctly.** `weight_decay` is not in `kwargs` or `group`, so it comes from `defaults=0`.

**Step 4 — check that group-exclusive keys are preserved.** A key present ONLY in the group (e.g., `layer_name`) should still appear in the resolved config.

In [ ]:
import torch as t

t.manual_seed(0)

def resolve_group_hparams(defaults, kwargs, group):
    # group > kwargs > defaults
    return {**defaults, **kwargs, **group}

# Defaults (lowest priority)
defaults = {'lr': 1e-2, 'weight_decay': 0.0, 'eps': 1e-8}

# Optimizer-level kwargs (medium priority)
kwargs = {'lr': 1e-3}

# Per-group overrides (highest priority)
group = {'lr': 1e-4, 'weight_decay': 1e-2, 'layer_name': 'embed'}

resolved = resolve_group_hparams(defaults, kwargs, group)
print(f"lr:           {resolved['lr']}")           # 1e-4 (group)
print(f"weight_decay: {resolved['weight_decay']}")  # 1e-2 (group)
print(f"eps:          {resolved['eps']}")           # 1e-8 (defaults)
print(f"layer_name:   {resolved['layer_name']}")    # 'embed' (group-only key)

assert resolved['lr'] == 1e-4
assert resolved['weight_decay'] == 1e-2
assert resolved['eps'] == 1e-8
assert resolved['layer_name'] == 'embed'
print("Group > kwarg > default precedence verified.")